# Bab 10: Regresi Linier dan Nonlinier

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab10/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab10")
print("Folder kerja:", os.getcwd())

## Listing 10.1: Regresi sederhana dari rumus

`kode/bab10/mini.py`

In [ ]:
import numpy as np

luas = np.array([75, 102, 59, 74, 40, 110])
omzet = np.array([1780, 1780, 1640, 1630, 1470, 1700])           # ribu Rp
xb, yb = luas.mean(), omzet.mean()
sxx = ((luas - xb) ** 2).sum()
sxy = ((luas - xb) * (omzet - yb)).sum()
b1 = sxy / sxx
b0 = yb - b1 * xb
yhat = b0 + b1 * luas
jkg = ((omzet - yhat) ** 2).sum()
jkt = ((omzet - yb) ** 2).sum()
print(f"x rata-rata {xb:.3f}, y rata-rata {yb:.3f}")
print(f"Sxx = {sxx:.2f}, Sxy = {sxy:.2f}")
print(f"b1 = {b1:.4f} ribu/m2, b0 = {b0:.2f} ribu")
print(f"JKG = {jkg:.1f}, JKT = {jkt:.1f}, R2 = {1 - jkg / jkt:.3f}")
print("sisa:", np.round(omzet - yhat, 1))
print("cek polyfit:", np.round(np.polyfit(luas, omzet, 1), 4))

## Listing 10.2: MLE dengan optimasi numerik

`kode/bab10/mle.py`

In [ ]:
import numpy as np
import statsmodels.formula.api as smf
from scipy.optimize import minimize
from gerai_tahunan import matang

X = np.c_[np.ones(len(matang)), matang[["luas_m2", "kota_besar"]]]
y = matang["omzet"].to_numpy()


def nll(theta):
    b, log_s = theta[:-1], theta[-1]
    s2 = np.exp(2 * log_s)
    e = y - X @ b
    return 0.5 * len(y) * np.log(2 * np.pi * s2) + (e @ e) / (2 * s2)


hasil = minimize(nll, x0=np.r_[y.mean(), 0, 0, np.log(y.std())],
                 method="BFGS")
ols = smf.ols("omzet ~ luas_m2 + kota_besar", matang).fit()
print("MLE :", hasil.x[:-1].round(3), " sigma", np.exp(hasil.x[-1]).round(2))
print("OLS :", ols.params.to_numpy().round(3),
      " sigma", np.sqrt(ols.ssr / len(y)).round(2))
print(f"log-likelihood maksimum: {-hasil.fun:.2f} (statsmodels "
      f"{ols.llf:.2f})")

## Listing 10.3: Regresi sederhana dan berganda untuk omzet gerai

`kode/bab10/berganda.py`

In [ ]:
import statsmodels.formula.api as smf
from gerai_tahunan import matang

print(f"{len(matang)} gerai, omzet harian rata-rata 2025 (ribu Rp)")
m1 = smf.ols("omzet ~ jumlah_kursi", matang).fit()
print(f"\nsederhana: kursi {m1.params['jumlah_kursi']:+.2f} "
      f"(p = {m1.pvalues['jumlah_kursi']:.2f}), R2 = {m1.rsquared:.3f}")
m2 = smf.ols("omzet ~ jumlah_kursi + luas_m2 + C(tipe_lokasi)"
             " + kota_besar", matang).fit()
tab = m2.summary2().tables[1][["Coef.", "Std.Err.", "t", "P>|t|"]]
tab.index = [i.replace("C(tipe_lokasi)[T.", "tipe ").replace("]", "")
             for i in tab.index]
print("\nberganda:")
print(tab.round(3).to_string())
print(f"R2 = {m2.rsquared:.3f}, R2 terkoreksi = {m2.rsquared_adj:.3f}, "
      f"sigma = {m2.mse_resid ** 0.5:.1f}")
print(f"uji F: F = {m2.fvalue:.1f}, p = {m2.f_pvalue:.1e}")

## Listing 10.4: Regresi log omzet harian dengan interaksi

`kode/bab10/harian.py`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA = "kode/data/"
h = pd.read_csv(DATA + "penjualan_harian.csv", parse_dates=["tanggal"])
g = pd.read_csv(DATA + "gerai.csv", parse_dates=["tanggal_buka"])
h = h.merge(g[["id_gerai", "tipe_lokasi", "tanggal_buka"]], on="id_gerai")
h = h[(h["tanggal"] < "2026-01-01") & (h["tutup_lebaran"] == 0)
      & (h["tanggal"] - h["tanggal_buka"] > pd.Timedelta(days=365))]
h["hari"] = h["tanggal"].dt.dayofweek
h["gajian"] = ((h["tanggal"].dt.day >= 25)
               | (h["tanggal"].dt.day <= 3)).astype(int)
h["tahun"] = h["tanggal"].dt.year - 2023
h["libur_kampus"] = ((h["tipe_lokasi"] == "kampus")
                     & (h["tanggal"].dt.month.isin([7, 8]))).astype(int)
rumus = ("np.log(omzet) ~ C(id_gerai) + C(hari) + tahun + gajian"
         " + hujan_mm + libur_kampus + libur_nasional"
         " + libur_nasional:C(tipe_lokasi) + ramadan"
         " + ramadan:C(tipe_lokasi)")
m = smf.ols(rumus, h).fit()
print(f"{len(h)} hari-gerai, {len(m.params)} koefisien, "
      f"R2 = {m.rsquared:.3f}")
efek = {"Sabtu vs Senin": "C(hari)[T.5]", "per tahun": "tahun",
        "gajian": "gajian", "hujan per 10 mm": "hujan_mm",
        "libur kampus (Jul-Agu)": "libur_kampus",
        "libur nasional, kampus": "libur_nasional",
        "Ramadan, kampus": "ramadan",
        "Ramadan, mal (tambahan)": "ramadan:C(tipe_lokasi)[T.mal]"}
for nama, k in efek.items():
    b = m.params[k] * (10 if "hujan" in nama else 1)
    print(f"{nama:24s} {100 * (np.exp(b) - 1):+6.1f}%")

lin = smf.ols(rumus.replace("np.log(omzet)", "omzet"), h).fit()
for nama, mm, pas in [("linier", lin, lin.fittedvalues),
                      ("log", m, np.exp(m.fittedvalues))]:
    kel = pd.qcut(pas, 5, labels=False)
    sd = mm.resid.groupby(kel).std() / mm.resid.std()
    print(f"sd sisa per kuintil ramalan, {nama:6s}:",
          " ".join(f"{v:.2f}" for v in sd))
sampel = h.sample(1500, random_state=0).index
pd.DataFrame({"pas": lin.fittedvalues[sampel] / 1000,
              "sisa": lin.resid[sampel] / 1000,
              "pas_log": m.fittedvalues[sampel],
              "sisa_log": m.resid[sampel]}).round(4).to_csv(
    "gambar/bab10/sisa_harian.dat", sep=" ", index=False)

## Listing 10.5: Perbandingan model, VIF, dan jarak Cook

`kode/bab10/diagnostik.py`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from gerai_tahunan import matang

rumus = "jumlah_kursi + luas_m2 + C(tipe_lokasi) + kota_besar"
lin = smf.ols("omzet ~ " + rumus, matang).fit()
log = smf.ols("np.log(omzet) ~ " + rumus, matang).fit()
kota = smf.ols("np.log(omzet) ~ jumlah_kursi + luas_m2"
               " + C(tipe_lokasi) + C(kota)", matang).fit()
for nama, m in [("linier", lin), ("log", log), ("log + kota", kota)]:
    print(f"{nama:10s}: R2 terkoreksi {m.rsquared_adj:.3f}, "
          f"AIC {m.aic:7.1f}")

X = lin.model.exog
for j, nama in enumerate(lin.model.exog_names):
    if j > 0:
        print(f"VIF {nama[:28]:28s} {variance_inflation_factor(X, j):.2f}")

pengaruh = kota.get_influence()
cook = pengaruh.cooks_distance[0]
i = np.argsort(-cook)[:3]
print("Cook terbesar:", ", ".join(f"{matang['id_gerai'].iloc[k]} "
                                  f"{cook[k]:.2f}" for k in i))
pd.DataFrame({"pas": np.r_[lin.fittedvalues],
              "sisa": np.r_[lin.resid],
              "pas_log": np.r_[log.fittedvalues],
              "sisa_log": np.r_[log.resid]}).round(4).to_csv(
    "gambar/bab10/sisa_gerai.dat", sep=" ", index=False)
print("sisa log + kota: maks", f"{np.abs(kota.resid).max():.3f}")

## Listing 10.6: Kurva pertumbuhan gerai baru

`kode/bab10/nonlinier.py`

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

DATA = "kode/data/"
h = pd.read_csv(DATA + "penjualan_harian.csv", parse_dates=["tanggal"])
g = pd.read_csv(DATA + "gerai.csv", parse_dates=["tanggal_buka"])
baru = g[g["tanggal_buka"].between("2023-04-01", "2024-03-31")]
h = h.merge(baru[["id_gerai", "tanggal_buka"]], on="id_gerai")
h = h[(h["tanggal"] < "2026-01-01") & (h["tutup_lebaran"] == 0)]
h["bulan"] = (h["tanggal"] - h["tanggal_buka"]).dt.days // 30.44
bln = h.groupby(["id_gerai", "bulan"])["omzet"].mean().reset_index()
acuan = bln[bln["bulan"].between(13, 20)].groupby("id_gerai")["omzet"].mean()
bln["relatif"] = bln["omzet"] / bln["id_gerai"].map(acuan)
bln = bln[bln["bulan"] <= 23]
t, r = bln["bulan"].to_numpy(), bln["relatif"].to_numpy()


def eksponen(t, a, tau):
    return 1 - a * np.exp(-t / tau)


def patah(t, a, t0):
    return np.where(t < t0, 1 - a * (1 - t / t0), 1.0)


pe, _ = curve_fit(eksponen, t, r, p0=[0.5, 6])
pp, _ = curve_fit(patah, t, r, p0=[0.5, 12])
p1 = np.polyfit(t, r, 2)
for nama, yhat, k in [("polinom kuadrat", np.polyval(p1, t), 3),
                      ("eksponen jenuh", eksponen(t, *pe), 2),
                      ("garis patah", patah(t, *pp), 2)]:
    jkg = ((r - yhat) ** 2).sum()
    print(f"{nama:16s} JKG {jkg:.3f}  RMSE {np.sqrt(jkg / len(r)):.3f}")
print(f"eksponen: a = {pe[0]:.3f}, tau = {pe[1]:.2f} bulan")
print(f"garis patah: a = {pp[0]:.3f}, t0 = {pp[1]:.2f} bulan")
grid = np.linspace(0, 23, 47)
pd.DataFrame({"t": grid, "eks": eksponen(grid, *pe),
              "patah": patah(grid, *pp), "poli": np.polyval(p1, grid)}
             ).round(4).to_csv("gambar/bab10/ramp_kurva.dat", sep=" ",
                               index=False)
bln[["bulan", "relatif"]].round(4).to_csv("gambar/bab10/ramp_titik.dat",
                                          sep=" ", index=False)

## Listing 10.7: Ridge dan lasso untuk data sedikit

`kode/bab10/regularisasi.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV, lasso_path
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan.csv")
fitur = ["umur", "frekuensi_90h", "frekuensi_90h_lalu", "recency_hari",
         "nilai_90h", "rata_nilai_struk", "porsi_promo", "porsi_aplikasi",
         "porsi_kopi_hitam", "porsi_pagi", "porsi_akhir_pekan",
         "ragam_menu", "keluhan_90h", "jarak_km", "pakai_ulasan"]
gerai = df[df["gerai_utama"] == "G24"]
X, y = gerai[fitur], gerai["belanja_bulan_depan"] / 1000
print(f"{len(gerai)} anggota gerai G24, 15 atribut")

cv = KFold(10, shuffle=True, random_state=0)
poli = PolynomialFeatures(2, include_bias=False)
model = {
    "OLS, 15 atribut": make_pipeline(StandardScaler(),
                                     LinearRegression()),
    "OLS, 135 atribut": make_pipeline(poli, StandardScaler(),
                                      LinearRegression()),
    "ridge, 135 atribut": make_pipeline(poli, StandardScaler(),
                                        RidgeCV(np.logspace(-2, 4, 60))),
    "lasso, 135 atribut": make_pipeline(poli, StandardScaler(),
                                        LassoCV(cv=5, max_iter=50000))}
for n, m in model.items():
    mae = -cross_val_score(m, X, y, cv=cv,
                           scoring="neg_mean_absolute_error").mean()
    print(f"{n:20s} MAE CV {mae:7.1f} ribu Rp")

m = model["lasso, 135 atribut"].fit(X, y)
koef = m[-1].coef_
nama = m[0].get_feature_names_out(fitur)
print(f"lasso memilih {(koef != 0).sum()} dari 135 atribut; terbesar:")
for i in np.argsort(-np.abs(koef))[:5]:
    print(f"  {nama[i]:30s} {koef[i]:+7.1f}")

Xs = StandardScaler().fit_transform(X)
alfa, jalur, _ = lasso_path(Xs, y - y.mean(), alphas=np.logspace(0.5, 2.6, 60))
pd.DataFrame(np.c_[np.log10(alfa), jalur.T], columns=["la"] + fitur
             ).round(3).to_csv("gambar/bab10/lasso.dat", sep=" ", index=False)

## Listing 10.8: Studi kasus omzet gerai baru

`kode/bab10/studi_kasus.py`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from gerai_tahunan import matang

rumus = {"linier": "omzet ~ luas_m2 + jumlah_kursi + C(tipe_lokasi)"
                   " + kota_besar",
         "log": "np.log(omzet) ~ luas_m2 + jumlah_kursi"
                " + C(tipe_lokasi) + kota_besar",
         "log+kota": "np.log(omzet) ~ luas_m2 + jumlah_kursi"
                     " + C(tipe_lokasi) + C(kota)"}
for nama, r in rumus.items():
    galat = []
    for i in range(len(matang)):
        latih = matang.drop(matang.index[i])
        p = smf.ols(r, latih).fit().predict(matang.iloc[[i]]).iloc[0]
        p = np.exp(p) if nama.startswith("log") else p
        galat.append(p / matang["omzet"].iloc[i] - 1)
    galat = np.abs(galat)
    print(f"{nama:8s} LOOCV: MAPE {galat.mean():.1%}, "
          f"galat terbesar {galat.max():.1%}")

m = smf.ols(rumus["log+kota"], matang).fit()
print("\nefek (persen):")
for k, v in m.params.items():
    if k != "Intercept":
        k2 = k.replace("C(tipe_lokasi)[T.", "tipe ").replace(
            "C(kota)[T.", "kota ").replace("]", "")
        print(f"  {k2:22s}"
              f" {100 * (np.exp(v) - 1):+6.1f}%  (p = {m.pvalues[k]:.3f})")

calon = pd.DataFrame({
    "lokasi": ["Pekanbaru, kampus", "Jakarta, perkantoran",
               "Medan, mal", "Padang, permukiman", "Batam, mal"],
    "luas_m2": [80, 70, 95, 110, 60],
    "jumlah_kursi": [40, 30, 35, 45, 28],
    "tipe_lokasi": ["kampus", "perkantoran", "mal", "permukiman", "mal"],
    "kota": ["Pekanbaru", "Jakarta", "Medan", "Padang", "Batam"]})
pr = m.get_prediction(calon).summary_frame(alpha=0.10)
calon["ramalan"] = np.exp(pr["mean"]).round(0)
calon["bawah90"] = np.exp(pr["obs_ci_lower"]).round(0)
calon["atas90"] = np.exp(pr["obs_ci_upper"]).round(0)
calon["thn1_juta"] = (calon["ramalan"] * 365 * 0.75 / 1000).round(0)
print("\nramalan omzet harian matang (ribu Rp), selang 90%:")
print(calon[["lokasi", "ramalan", "bawah90", "atas90",
             "thn1_juta"]].to_string(index=False))